In [ ]:
# Lighting robustness evaluation using the persisted SVM pipeline from steelblast_svm_glcm_dwt.ipynb
import json
from pathlib import Path

import joblib
import numpy as np
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline

from steelblast_classic_features import (
    FeatureExtractionConfig,
    extract_features_from_image,
    load_split,
    normalize_illumination,
    read_grayscale_image,
)

# parameters for bias analysis
dataset_dir = Path("doi-10.34894-ekznn0(1)/SteelBlastQC")
model_path = Path("steelblast_svm_glcm_dwt.joblib")

def perturb_lighting(image: np.ndarray, perturbation: str) -> np.ndarray:
    if perturbation == "baseline":
        return image
    if perturbation == "dark_25":
        return np.clip(image * 0.75, 0.0, 1.0)
    if perturbation == "dark_50":
        return np.clip(image * 0.50, 0.0, 1.0)
    if perturbation == "bright_25":
        return np.clip(image * 1.25, 0.0, 1.0)
    if perturbation == "bright_50":
        return np.clip(image * 1.50, 0.0, 1.0)
    if perturbation == "offset_plus_10":
        return np.clip(image + 0.10, 0.0, 1.0)
    if perturbation == "offset_minus_10":
        return np.clip(image - 0.10, 0.0, 1.0)
    if perturbation == "low_contrast":
        return np.clip((image - 0.5) * 0.75 + 0.5, 0.0, 1.0)
    if perturbation == "high_contrast":
        return np.clip((image - 0.5) * 1.25 + 0.5, 0.0, 1.0)
    raise ValueError(f"Unknown lighting perturbation: {perturbation}")


def build_perturbed_feature_matrix(
    image_paths: list[Path],
    feature_config: FeatureExtractionConfig,
    perturbation: str,
) -> np.ndarray:
    features = []
    for image_path in image_paths:
        raw_image = read_grayscale_image(image_path, feature_config.image_size, config=None)
        perturbed_image = perturb_lighting(raw_image, perturbation)
        normalized_image = normalize_illumination(perturbed_image, feature_config)
        features.append(extract_features_from_image(normalized_image, feature_config))
    return np.vstack(features)


def evaluate_lighting_robustness(
    image_paths: list[Path],
    y_true: np.ndarray,
    baseline_predictions: np.ndarray,
    fitted_model: Pipeline,
    feature_config: FeatureExtractionConfig,
) -> list[dict[str, object]]:
    perturbations = [
        "baseline",
        "dark_25",
        "dark_50",
        "offset_plus_10",
        "offset_minus_10",
        "low_contrast",
        "high_contrast",
        "bright_25",
        "bright_50",
    ]
    normal_lighting_tests = {
        "dark_25",
        "dark_50",
        "offset_plus_10",
        "offset_minus_10",
        "low_contrast",
        "high_contrast",
    }
    rows = []

    for perturbation in perturbations:
        X_variant = build_perturbed_feature_matrix(
            image_paths,
            feature_config,
            perturbation,
        )
        y_variant = fitted_model.predict(X_variant)
        matrix_variant = confusion_matrix(y_true, y_variant, labels=[0, 1])
        accuracy = float(np.mean(y_variant == y_true))
        flips = int(np.sum(y_variant != baseline_predictions))
        good_recall = float(matrix_variant[0, 0] / matrix_variant[0].sum())
        not_good_recall = float(matrix_variant[1, 1] / matrix_variant[1].sum())
        rows.append(
            {
                "perturbation": perturbation,
                "accuracy": accuracy,
                "prediction_flips_vs_baseline": flips,
                "flip_rate_vs_baseline": float(flips / len(y_true)),
                "good_recall": good_recall,
                "not_good_recall": not_good_recall,
                "confusion_matrix": matrix_variant.tolist(),
                "included_in_robustness_claim": perturbation in normal_lighting_tests,
            }
        )

    return rows


feature_config = FeatureExtractionConfig(
    image_size=256,
    illumination_normalization="clahe",
    clahe_clip_limit=0.01,
    glcm_levels=32,
    glcm_distances=(1, 2, 4, 8),
    glcm_angles=(0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4),
    glcm_properties=("contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"),
    dwt_wavelet="db2",
    dwt_level=3,
)



if not model_path.exists():
    raise FileNotFoundError(f"Saved model not found: {model_path.resolve()}")

fitted_model: Pipeline = joblib.load(model_path)
test_paths, y_test = load_split(dataset_dir, "test")
X_test_baseline = build_perturbed_feature_matrix(test_paths, feature_config, "baseline")
y_pred_baseline = fitted_model.predict(X_test_baseline)

lighting_robustness = evaluate_lighting_robustness(
    test_paths,
    y_test,
    y_pred_baseline,
    fitted_model,
    feature_config,
)

print("Loaded model:", model_path.resolve())
print("Lighting robustness check")
print("perturbation       acc   flips  flip_rate  good_recall  not_good_recall")
for row in lighting_robustness:
    print(
        f"{row['perturbation']:16s} "
        f"{row['accuracy']:.3f} "
        f"{row['prediction_flips_vs_baseline']:5d} "
        f"{row['flip_rate_vs_baseline']:.3f} "
        f"{row['good_recall']:.3f} "
        f"{row['not_good_recall']:.3f}"
    )

claim_rows = [row for row in lighting_robustness if row["included_in_robustness_claim"]]
min_claim_accuracy = min(row["accuracy"] for row in claim_rows)
max_claim_flip_rate = max(row["flip_rate_vs_baseline"] for row in claim_rows)
lighting_robustness_summary = {
    "claim": "Model is robust to non-saturating lighting variation after CLAHE normalization.",
    "criterion": "For darkening, additive offsets, and moderate contrast shifts: accuracy >= 0.90 and flip_rate <= 0.06.",
    "passed": bool(min_claim_accuracy >= 0.90 and max_claim_flip_rate <= 0.06),
    "min_claim_accuracy": float(min_claim_accuracy),
    "max_claim_flip_rate": float(max_claim_flip_rate),
    "note": "Severe brightening is reported as a stress test because clipping can destroy texture information.",
}

print("\nRobustness summary")
print(json.dumps(lighting_robustness_summary, indent=2))

metrics_path = Path("bias_analysis_metrics.json")
if metrics_path.exists():
    metrics_with_robustness = json.loads(metrics_path.read_text())
else:
    metrics_with_robustness = {}

metrics_with_robustness["model_path"] = str(model_path)
metrics_with_robustness["lighting_robustness"] = {
    "summary": lighting_robustness_summary,
    "results": lighting_robustness,
}
metrics_path.write_text(json.dumps(metrics_with_robustness, indent=2))
print(f"Saved lighting robustness results to: {metrics_path.resolve()}")
